In [1]:
# Replace 'your_filename.zip' with the actual name you uploaded
!unzip archive.zip

Archive:  archive.zip
  inflating: NEU-DET/train/annotations/crazing_158.xml  
  inflating: NEU-DET/train/annotations/crazing_159.xml  
  inflating: NEU-DET/train/annotations/crazing_16.xml  
  inflating: NEU-DET/train/annotations/crazing_160.xml  
  inflating: NEU-DET/train/annotations/crazing_161.xml  
  inflating: NEU-DET/train/annotations/crazing_162.xml  
  inflating: NEU-DET/train/annotations/crazing_163.xml  
  inflating: NEU-DET/train/annotations/crazing_164.xml  
  inflating: NEU-DET/train/annotations/crazing_165.xml  
  inflating: NEU-DET/train/annotations/crazing_166.xml  
  inflating: NEU-DET/train/annotations/crazing_167.xml  
  inflating: NEU-DET/train/annotations/crazing_168.xml  
  inflating: NEU-DET/train/annotations/crazing_169.xml  
  inflating: NEU-DET/train/annotations/crazing_17.xml  
  inflating: NEU-DET/train/annotations/crazing_170.xml  
  inflating: NEU-DET/train/annotations/crazing_171.xml  
  inflating: NEU-DET/train/annotations/crazing_172.xml  
  inflating

In [2]:
# The main folder that was extracted
base_dir = 'NEU-DET'

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Setup training and validation generators
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True
)

# NEU-DET is already split into train/validation folders in your zip
train_generator = datagen.flow_from_directory(
    'NEU-DET/train/images', # Point to the train images
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = datagen.flow_from_directory(
    'NEU-DET/validation/images', # Point to the validation images
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

Found 1440 images belonging to 6 classes.
Found 360 images belonging to 6 classes.


In [3]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models

# Load ResNet50
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# Custom Head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [4]:
history = model.fit(train_generator, validation_data=val_generator, epochs=10)

Epoch 1/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 42s 642ms/step - accuracy: 0.1986 - loss: 1.7981 - val_accuracy: 0.4667 - val_loss: 1.7572
Epoch 2/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 22s 492ms/step - accuracy: 0.2611 - loss: 1.7422 - val_accuracy: 0.3333 - val_loss: 1.7018
Epoch 3/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 22s 486ms/step - accuracy: 0.2799 - loss: 1.7169 - val_accuracy: 0.3444 - val_loss: 1.6700
Epoch 4/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 468ms/step - accuracy: 0.3313 - loss: 1.6387 - val_accuracy: 0.3500 - val_loss: 1.6158
Epoch 5/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 22s 489ms/step - accuracy: 0.3264 - loss: 1.5978 - val_accuracy: 0.2722 - val_loss: 1.6394
Epoch 6/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 22s 490ms/step - accuracy: 0.3889 - loss: 1.5388 - val_accuracy: 0.2861 - val_loss: 1.5363
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 21s 472ms/step - accuracy: 0.3833 - loss: 1.5013 - val_accuracy: 0.4111 - val_loss: 1.4783
Epoch 8/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 42s 485ms/step - accuracy: 0.4090 - loss: 1.4611 - val_accu

In [5]:
import numpy as np
from sklearn.metrics import classification_report

# Predict the classes for the validation set
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)

# Get the true labels
y_true = val_generator.classes

# Print the report
print(classification_report(y_true, y_pred, target_names=val_generator.class_indices.keys()))

12/12 ━━━━━━━━━━━━━━━━━━━━ 13s 614ms/step
                 precision    recall  f1-score   support

        crazing       0.20      0.28      0.23        60
      inclusion       0.12      0.10      0.11        60
        patches       0.14      0.10      0.12        60
 pitted_surface       0.20      0.57      0.30        60
rolled-in_scale       0.00      0.00      0.00        60
      scratches       0.33      0.05      0.09        60

       accuracy                           0.18       360
      macro avg       0.16      0.18      0.14       360
   weighted avg       0.16      0.18      0.14       360



In [7]:
def get_factory_decision(prediction_array):
    # Get the index of the highest probability
    class_idx = np.argmax(prediction_array)
    probability = np.max(prediction_array)

    # Logic: If confidence is high (> 85%), take action
    if probability >= 0.85:
        return f"ACTION: Trigger rejection for {list(val_generator.class_indices.keys())[class_idx]}"
    else:
        return "DECISION: Product passed (Low defect confidence)"

# Test with a single image from the validation set
# Use the Python built-in next() function
img, label = next(val_generator)

# Now run the prediction
sample_prediction = model.predict(img[0:1])
print(get_factory_decision(sample_prediction))
sample_prediction = model.predict(img[0:1])
print(get_factory_decision(sample_prediction))

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
DECISION: Product passed (Low defect confidence)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
DECISION: Product passed (Low defect confidence)
